In [1]:
# Tanggal: 25 September 2025
# NIM: 25630300009
# Nama: Lefi Andri Lestari
print("Latihan 3 DDDM Minggu 2 - Analisis data Titanic")
print("NIM: 25630300009 | Nama: Lefi Andri Lestari")


Latihan 3 DDDM Minggu 2 - Analisis data Titanic
NIM: 25630300009 | Nama: Lefi Andri Lestari


# Latihan 3 DDDM Minggu 2

**Tujuan:** (1) mendeskripsikan karakteristik penumpang Titanic dan (2) menghitung korelasi Pearson antara tarif tiket (`Fare`) dan status selamat (`Survived`). Notebook ini menggunakan dataset persis dari tautan pada handout. Jalankan **Runtime → Run all** di Google Colab untuk menghitung ulang seluruh hasil.

> Jika Colab tidak dapat mengunduh CSV, sel pemuatan data akan meminta Anda mengunggah `titanic.csv` dari tautan yang sama. Angka jawaban di bagian akhir berlaku untuk dataset 891 baris yang dianalisis di sini.


In [2]:
import io
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv"
try:
    df = pd.read_csv(DATA_URL)
    print("Dataset dimuat dari tautan pada handout.")
except Exception as error:
    print("Unduhan otomatis gagal:", error)
    print("Unggah file titanic.csv dari tautan pada handout.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Tidak ada file CSV yang diunggah.")
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))

required = {"PassengerId", "Survived", "Pclass", "Sex", "Age", "Fare"}
if not required.issubset(df.columns):
    raise ValueError("Kolom wajib tidak tersedia; periksa berkas CSV.")
print("Ukuran data (baris, kolom):", df.shape)
print(df[["PassengerId", "Survived", "Pclass", "Sex", "Age", "Fare"]]
      .head().to_string(index=False))


Dataset dimuat dari tautan pada handout.
Ukuran data (baris, kolom): (891, 12)
 PassengerId  Survived  Pclass    Sex  Age    Fare
           1         0       3   male 22.0  7.2500
           2         1       1 female 38.0 71.2833
           3         1       3 female 26.0  7.9250
           4         1       1 female 35.0 53.1000
           5         0       3   male 35.0  8.0500


## 1. Pemeriksaan data

Setiap baris mewakili satu penumpang. `Survived` bernilai **0** untuk tidak selamat dan **1** untuk selamat. `Pclass` menunjukkan kelas tiket 1, 2, atau 3. Nilai kosong perlu diperiksa sebelum menafsirkan statistik.


In [3]:
print(f"Jumlah penumpang: {len(df)}")
print(f"Jumlah variabel: {df.shape[1]}")
print("\nNilai kosong per variabel:")
print(df.isna().sum().to_string())
print("\nTipe data variabel utama:")
print(df[["Survived", "Pclass", "Sex", "Age", "Fare"]]
      .dtypes.to_string())


Jumlah penumpang: 891
Jumlah variabel: 12

Nilai kosong per variabel:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2

Tipe data variabel utama:
Survived      int64
Pclass        int64
Sex          object
Age         float64
Fare        float64


## 2. Karakteristik penumpang

Ringkasan berikut mencakup usia, jenis kelamin, kelas tiket, tarif, dan status selamat. Rata-rata usia memakai hanya baris yang mempunyai nilai `Age`. Persentase jenis kelamin, kelas, dan status selamat memakai seluruh 891 baris.


In [4]:
print("RINGKASAN NUMERIK")
print(df[["Age", "Fare"]].describe().round(2).to_string())
print("\nJENIS KELAMIN")
sex_summary = pd.DataFrame({
    "Jumlah": df["Sex"].value_counts(),
    "Persentase": (df["Sex"].value_counts(normalize=True) * 100).round(2),
})
print(sex_summary.to_string())
print("\nKELAS TIKET")
class_summary = pd.DataFrame({
    "Jumlah": df["Pclass"].value_counts().sort_index(),
    "Persentase": (df["Pclass"].value_counts(normalize=True)
                   .sort_index() * 100).round(2),
})
print(class_summary.to_string())


RINGKASAN NUMERIK
          Age    Fare
count  714.00  891.00
mean    29.70   32.20
std     14.53   49.69
min      0.42    0.00
25%     20.12    7.91
50%     28.00   14.45
75%     38.00   31.00
max     80.00  512.33

JENIS KELAMIN
        Jumlah  Persentase
Sex                       
male       577       64.76
female     314       35.24

KELAS TIKET
        Jumlah  Persentase
Pclass                    
1          216       24.24
2          184       20.65
3          491       55.11


In [5]:
print("STATUS KESELAMATAN")
status_summary = pd.DataFrame({
    "Jumlah": df["Survived"].value_counts().sort_index(),
    "Persentase": (df["Survived"].value_counts(normalize=True)
                   .sort_index() * 100).round(2),
})
status_summary.index = ["Tidak selamat (0)", "Selamat (1)"]
print(status_summary.to_string())
print(f"\nUsia tersedia: {df['Age'].count()} dari {len(df)} penumpang")
print(f"Rata-rata usia: {df['Age'].mean():.2f} tahun")
print(f"Median usia: {df['Age'].median():.2f} tahun")
print(f"Rata-rata Fare: {df['Fare'].mean():.2f}")
print(f"Median Fare: {df['Fare'].median():.4f}")


STATUS KESELAMATAN
                   Jumlah  Persentase
Tidak selamat (0)     549       61.62
Selamat (1)           342       38.38

Usia tersedia: 714 dari 891 penumpang
Rata-rata usia: 29.70 tahun
Median usia: 28.00 tahun
Rata-rata Fare: 32.20
Median Fare: 14.4542


## 3. Korelasi `Fare` dengan `Survived`

Pearson `r` mengukur arah dan kekuatan **hubungan linear**. Karena `Survived` dikodekan sebagai 0 dan 1, korelasi Pearson dapat dihitung untuk `Fare` dan `Survived`. Kode `dropna()` memastikan keduanya tersedia pada baris yang sama. Nilai `r` berada antara −1 dan +1; angka positif menunjukkan kecenderungan searah.


In [6]:
pasangan = df[["Fare", "Survived"]].dropna()
r = pasangan["Fare"].corr(pasangan["Survived"], method="pearson")
r_matriks = pasangan.corr(method="pearson").loc["Fare", "Survived"]
assert abs(r - r_matriks) < 1e-12

print(f"Jumlah pasangan data lengkap: {len(pasangan)}")
print(f"Korelasi Pearson Fare dan Survived (r): {r:.6f}")
print("\nRata-rata Fare menurut status selamat:")
fare_per_status = pasangan.groupby("Survived")["Fare"].agg(["count", "mean", "median"])
print(fare_per_status.round(2).to_string())


Jumlah pasangan data lengkap: 891
Korelasi Pearson Fare dan Survived (r): 0.257307

Rata-rata Fare menurut status selamat:
          count   mean  median
Survived                      
0           549  22.12    10.5
1           342  48.40    26.0


### Kriteria interpretasi kekuatan korelasi

Tabel di bawah mengikuti rentang yang diberikan dalam soal. Rentangnya dinyatakan dengan **dua angka desimal**. Karena itu, kode klasifikasi membulatkan hasil `r` ke dua angka desimal dahulu, sementara nilai `r` asli tetap ditampilkan untuk pelaporan yang lebih teliti.

| Rentang positif | Kategori | Rentang negatif | Kategori |
| --- | --- | --- | --- |
| +0,80 sampai +1,00 | Very Strong Positive | −1,00 sampai −0,80 | Very Strong Negative |
| +0,60 sampai +0,79 | Strong Positive | −0,79 sampai −0,60 | Strong Negative |
| +0,40 sampai +0,59 | Moderate Positive | −0,59 sampai −0,40 | Moderate Negative |
| +0,20 sampai +0,39 | Weak Positive | −0,39 sampai −0,20 | Weak Negative |
| +0,01 sampai +0,19 | Very Weak Positive | −0,19 sampai −0,01 | Very Weak Negative |

**Catatan:** Nilai tepat 0,00 ditafsirkan tanpa arah korelasi linear, walaupun rentang positif pada tabel asal tertulis mulai 0,00. Pengelompokan ini adalah aturan interpretasi yang disediakan soal, bukan bukti sebab akibat.


In [7]:
def kategori_korelasi(koefisien):
    """Klasifikasi menurut tabel soal pada ketelitian dua desimal."""
    if pd.isna(koefisien):
        return "Tidak dapat dihitung"

    r_dua_desimal = round(float(koefisien), 2)
    if r_dua_desimal == 0:
        return "Tidak ada arah korelasi linear (r dibulatkan = 0,00)"

    nilai_absolut = abs(r_dua_desimal)
    if nilai_absolut >= 0.80:
        kekuatan = "Very Strong"
    elif nilai_absolut >= 0.60:
        kekuatan = "Strong"
    elif nilai_absolut >= 0.40:
        kekuatan = "Moderate"
    elif nilai_absolut >= 0.20:
        kekuatan = "Weak"
    else:
        kekuatan = "Very Weak"

    arah = "Positive" if r_dua_desimal > 0 else "Negative"
    return f"{kekuatan} {arah}"

kategori = kategori_korelasi(r)
print(f"Nilai r asli: {r:+.6f}")
print(f"Nilai r untuk klasifikasi (2 desimal): {r:+.2f}")
print(f"Kategori menurut tabel: {kategori}")


Nilai r asli: +0.257307
Nilai r untuk klasifikasi (2 desimal): +0.26
Kategori menurut tabel: Weak Positive


## 4. Jawaban tertulis

### Pertanyaan 1  Karakteristik penumpang

Dataset ini memuat **891 penumpang dan 12 variabel**. Terdapat **577 laki-laki (64,76%)** dan **314 perempuan (35,24%)**. Kelas tiket terbanyak adalah **kelas 3: 491 penumpang (55,11%)**; kelas 1 berjumlah 216 (24,24%) dan kelas 2 berjumlah 184 (20,65%). Usia tersedia untuk **714 penumpang**, sedangkan 177 catatan usia kosong. Rata-rata usia yang terisi sekitar **29,70 tahun**, dengan median **28 tahun**. Tarif tiket (`Fare`) rata-rata **32,20** dan mediannya **14,4542** pada satuan tarif dalam dataset. Sebanyak **342 penumpang (38,38%)** tercatat selamat dan **549 (61,62%)** tidak selamat. Variabel `Cabin` memiliki 687 nilai kosong dan `Embarked` memiliki 2 nilai kosong; kelengkapan tiap variabel perlu diperhatikan jika analisis diperluas.

### Pertanyaan 2  Korelasi tarif tiket dan keselamatan

Hasil perhitungan Pearson untuk **891 pasangan data lengkap** adalah **r = +0,257307** (sekitar **+0,2573**). Nilai tersebut dibulatkan menjadi **+0,26** dan masuk rentang **+0,20 sampai +0,39**, sehingga menurut tabel kriteria termasuk **Weak Positive (korelasi positif lemah)**. Artinya, terdapat hubungan linear positif yang lemah: dalam dataset ini, penumpang dengan tarif tiket lebih tinggi cenderung lebih sering tercatat selamat. Sebagai gambaran tambahan, rata-rata `Fare` kelompok selamat adalah **48,40**, sedangkan kelompok tidak selamat **22,12**. Korelasi tersebut **tidak membuktikan bahwa membayar tiket lebih mahal menyebabkan keselamatan**. Kelas tiket dan faktor lain dapat berkaitan dengan tarif sekaligus peluang selamat. Kesimpulan berlaku untuk data yang dianalisis, bukan untuk semua penumpang Titanic.

**Sumber data:** [pandas Titanic CSV](https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv), sebagaimana tercantum dalam handout DDDM W2.
